# 4. Region of Convergence

In this notebook, we find the region of convergence according to the theoretical bounds.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using LinkedMasses

## Region of Convergence

According to the theoretical analysis, the absolute value of $|a_\theta|$ must be less than $2 \frac{c_0}{m_0}$. A conservative bound on the function $a_\theta$ is given by

$$ |a_\theta| \leq \frac{|m - \varepsilon(m + m_0)| + \|\tilde{\zeta}\|}{2 m_0 L (1 - \varepsilon)} \|\tilde{v}_\varepsilon\| + \frac{\sqrt{2} \|\tilde{\zeta}\|}{m_0} $$

In [ ]:
function a_θ(v_err::Real, ζ_err::Real, ctrl::AdaptiveLinearizingController)
    ε = ctrl.ε
    m0 = ctrl.model.m0
    m = ctrl.model.m
    L = ctrl.model.L
    
    term1 = (abs(m - ε * (m + m0)) + ζ_err) / (2 * m0 * L * (1 - ε)) * v_err
    term2 = sqrt(2) * ζ_err / m0
    
    return term1 + term2
end

## Lyapunov Regions

Consider the Lyapunov function

$$ V(q, \xi) = k_v\! \left(\! \sqrt{\Delta^2 + e_x^2 + e_y^2} - \Delta \right) + \frac{1}{2} b_v \tilde{v}_{\varepsilon}^\top \bar{M}(q) \tilde{v}_{\varepsilon} + \frac{1}{2\gamma} b_v \|\tilde{\zeta}\|^2 $$

where

$$ \bar{M} = (m + m_0) I -  \left(m + \frac{\varepsilon m_0}{\varepsilon - 1}\right) \Gamma^\perp (\Gamma^\perp)^\top $$

is the "inertia" matrix of the control point, and $b_v$ is a positive constant such that

$$ b_v >  \frac{1}{4 U \lambda_{\min}}$$

where $\lambda_{\min}$ is the smallest eigenvalue of $\bar{M}$. Note that $\bar{M}$ is a positive definite matrix with eigenvalues $\lambda_1 = m + m_0$ and $\lambda_2 = \frac{m_0}{1 - \varepsilon}$.

In [ ]:
function λ(ctrl::AdaptiveLinearizingController)
    m0 = ctrl.model.m0
    m = ctrl.model.m
    
    λ1 = m + m0
    λ2 = m0 / (1 - ctrl.ε)

    return λ1, λ2
end

λ_min(ctrl::AdaptiveLinearizingController) = minimum(λ(ctrl))
λ_max(ctrl::AdaptiveLinearizingController) = maximum(λ(ctrl))

The Lyapunov function is positive definite and decrescent. Specifically, it satisfies

$$ W_1(\xi) \leq V(q, \xi) \leq W_2(\xi) $$

where

$$ W_1(\xi) = k_v\left(\sqrt{\Delta^2 + \|e\|^2} - \Delta\right) + \frac{1}{2} b_v \lambda_{\min} \|\tilde{v}_\varepsilon\|^2 + \frac{1}{2\gamma} b_v \|\tilde{\zeta}\|^2 $$

$$ W_2(\xi) = k_v\left(\sqrt{\Delta^2 + \|e\|^2} - \Delta\right) + \frac{1}{2} b_v \lambda_{\max} \|\tilde{v}_\varepsilon\|^2 + \frac{1}{2\gamma} b_v \|\tilde{\zeta}\|^2 $$

The derivative of $V$ is negative semidefinite. Let $\Omega_1 = \left\{ \xi \mid W_1(\xi) \leq \alpha \right\}$ and $\Omega_2 = \left\{ \xi \mid W_2(\xi) \leq \alpha \right\}$ be the sublevel sets of $W_1$ and $W_2$. Since $\dot{V} \leq 0$, any trajectory of the system starting in $\Omega_2$ remains in $\Omega_1$.

To find the region of convergence, we need to find the largest value of $\alpha$ such that the set $\Omega_1$ satisfies $|a_\theta(\xi)| < 2 c_0 / m_0$. In other words, we need to solve the optimization problem:

$$ \alpha = \max \alpha $$
$$ \text{subject to } |a_\theta(\xi)| < 2 c_0 / m_0 \quad \forall \xi \in \Omega_1 $$

This problem is equivalent to finding the smallest value of $W_1(\xi)$ for which $|a_\theta(\xi)| = 2 c_0 / m_0$, i.e.

$$ \alpha = \min_{\xi} W_1(\xi) $$
$$ \text{subject to } |a_\theta(\xi)| = 2 c_0 / m_0 $$

In [ ]:
function b_v(ctrl::AdaptiveLinearizingController)
    U = ctrl.los.U
    return 1 / (U * λ_min(ctrl))
end

function W1(e::Real, v_err::Real, ζ_err::Real, ctrl::AdaptiveLinearizingController)
    k_v = ctrl.k_v
    Δ = ctrl.los.Δ
    b = b_v(ctrl)

    W_e = k_v * (sqrt(e^2 + Δ^2) - Δ)
    W_v = 0.5 * b * λ_min(ctrl) * v_err^2
    W_ζ = 0.5 * b / ctrl.k_a * ζ_err^2
    return W_e + W_v + W_ζ
end

function W2(e::Real, v_err::Real, ζ_err::Real, ctrl::AdaptiveLinearizingController)
    k_v = ctrl.k_v
    Δ = ctrl.los.Δ
    b = b_v(ctrl)

    W_e = k_v * (sqrt(e^2 + Δ^2) - Δ)
    W_v = 0.5 * b * λ_max(ctrl) * v_err^2
    W_ζ = 0.5 * b / ctrl.k_a * ζ_err^2
    return W_e + W_v + W_ζ
end

In [ ]:
# Parameters of the actual system
V_c = [-0.2, 0.0]

m0 = 200.0
m = 40.0
c0 = 250.0
c = 50.0
L = 10.0

params = LinkedMassesParameters(L, m0, m, c0, c)

ε = 0.7

# Path
R = 10.0
ϕ0 = -π / 2
p_origin = [2.0, 12.0]
path_circle(s) = p_origin + R * [cos(ϕ0 + s), sin(ϕ0 + s)]

# Guidance parameters
U = 1.0
Δ = 5.0
los = LOSParameters(U, Δ)

# Controller gains
k_v = 0.1
γ = 5.0

ctrl = AdaptiveLinearizingController(params, los, path_circle, k_v, γ, ε, V_c)

In [ ]:
## Find the region of convergence
using JuMP, Ipopt

model = Model(Ipopt.Optimizer)
@variable(model, e >= 0)
@variable(model, v_err >= 0)
@variable(model, ζ_err >= 0)

# min W1
register(model, :W1, 3, (_e, _v, _ζ) -> W1(_e, _v, _ζ, ctrl), autodiff = true)
@NLobjective(model, Min, W1(e, v_err, ζ_err))

# subject to a_θ = 2c0 / m0
register(model, :a_θ, 2, (_v, _ζ) -> a_θ(_v, _ζ, ctrl), autodiff = true)
@NLconstraint(model, a_θ(v_err, ζ_err) == 2c0 / m0)

optimize!(model)
α_opt = objective_value(model)

Now, let's visualize the regions in the $\tilde{v}$-$\tilde{\zeta}$ plane:

In [ ]:
function find_ζ_aθ(v_err::Real, ctrl::AdaptiveLinearizingController)
    # For given v_err, find ζ_err such that a_theta == 2c0/m0
    ε = ctrl.ε
    m0 = ctrl.model.m0
    m = ctrl.model.m
    L = ctrl.model.L
    
    rhs = 2 * ctrl.model.c0 / m0
    c1 = abs(m - ε * (m + m0)) / (2 * m0 * L * (1 - ε)) * v_err
    k1 = v_err / (2 * m0 * L * (1 - ε))
    k2 = sqrt(2) / m0
    # Solve for ζ_err: c1 + k1 * ζ_err + k2 * ζ_err == rhs
    ζ_err = (rhs - c1) / (k1 + k2)
    return ζ_err
end

function find_ζ_W1(e::Real, v_err::Real, α::Real, ctrl::AdaptiveLinearizingController)
    # For given e and v_err, find ζ_err such that W1 == α
    k_v = ctrl.k_v
    Δ = ctrl.los.Δ
    b = b_v(ctrl)

    W_e = k_v * (sqrt(e^2 + Δ^2) - Δ)
    W_v = 0.5 * b * λ_min(ctrl) * v_err^2
    W_ζ = α - W_e - W_v
    if W_ζ < 0
        return NaN
    end
    ζ_err = sqrt(2 * W_ζ * ctrl.k_a / b)
    return ζ_err
end

function find_v_W1(e::Real, ζ_err::Real, α::Real, ctrl::AdaptiveLinearizingController)
    # For given e and ζ_err, find v_err such that W1 == α
    k_v = ctrl.k_v
    Δ = ctrl.los.Δ
    b = b_v(ctrl)

    W_e = k_v * (sqrt(e^2 + Δ^2) - Δ)
    W_ζ = 0.5 * b / ctrl.k_a * ζ_err^2
    W_v = α - W_e - W_ζ
    if W_v < 0
        return NaN
    end
    v_err = sqrt(2 * W_v / (b * λ_min(ctrl)))
    return v_err    
end

function find_e_W1(v_err::Real, ζ_err::Real, α::Real, ctrl::AdaptiveLinearizingController)
    # For given v_err and ζ_err, find e such that W1 == α
    k_v = ctrl.k_v
    Δ = ctrl.los.Δ
    b = b_v(ctrl)

    W_v = 0.5 * b * λ_min(ctrl) * v_err^2
    W_ζ = 0.5 * b / ctrl.k_a * ζ_err^2
    W_e = α - W_v - W_ζ
    if W_e < 0
        return NaN
    end
    e = sqrt(((W_e / k_v) + Δ)^2 - Δ^2)
    return e    
end

function W1_limits(α::Real, ctrl::AdaptiveLinearizingController)
    e_max = find_e_W1(0.0, 0.0, α, ctrl)
    v_err_max = find_v_W1(0.0, 0.0, α, ctrl)
    ζ_err_max = find_ζ_W1(0.0, 0.0, α, ctrl)
    return e_max, v_err_max, ζ_err_max
end

function find_ζ_W2(e::Real, v_err::Real, α::Real, ctrl::AdaptiveLinearizingController)
    # For given e and v_err, find ζ_err such that W2 == α
    k_v = ctrl.k_v
    Δ = ctrl.los.Δ
    b = b_v(ctrl)

    W_e = k_v * (sqrt(e^2 + Δ^2) - Δ)
    W_v = 0.5 * b * λ_max(ctrl) * v_err^2
    W_ζ = α - W_e - W_v
    if W_ζ < 0
        return NaN
    end
    ζ_err = sqrt(2 * W_ζ * ctrl.k_a / b)
    return ζ_err
end

function find_v_W2(e::Real, ζ_err::Real, α::Real, ctrl::AdaptiveLinearizingController)
    # For given e and ζ_err, find v_err such that W2 == α
    k_v = ctrl.k_v
    Δ = ctrl.los.Δ
    b = b_v(ctrl)

    W_e = k_v * (sqrt(e^2 + Δ^2) - Δ)
    W_ζ = 0.5 * b / ctrl.k_a * ζ_err^2
    W_v = α - W_e - W_ζ
    if W_v < 0
        return NaN
    end
    v_err = sqrt(2 * W_v / (b * λ_max(ctrl)))
    return v_err    
end

function find_e_W2(v_err::Real, ζ_err::Real, α::Real, ctrl::AdaptiveLinearizingController)
    # For given v_err and ζ_err, find e such that W2 == α
    k_v = ctrl.k_v
    Δ = ctrl.los.Δ
    b = b_v(ctrl)

    W_v = 0.5 * b * λ_max(ctrl) * v_err^2
    W_ζ = 0.5 * b / ctrl.k_a * ζ_err^2
    W_e = α - W_v - W_ζ
    if W_e < 0
        return NaN
    end
    e = sqrt(((W_e / k_v) + Δ)^2 - Δ^2)
    return e    
end

function W2_limits(α::Real, ctrl::AdaptiveLinearizingController)
    e_max = find_e_W2(0.0, 0.0, α, ctrl)
    v_err_max = find_v_W2(0.0, 0.0, α, ctrl)
    ζ_err_max = find_ζ_W2(0.0, 0.0, α, ctrl)
    return e_max, v_err_max, ζ_err_max
end

In [ ]:
## Plot the sets in the v_err - ζ_err plane
using Plots, LaTeXStrings
using CSV, DataFrames

# Heatmap of a_θ
_, v_err_max, ζ_err_max = W1_limits(α_opt, ctrl)
v_errs = range(0.0, v_err_max, length=24)
ζ_errs = range(0.0, max(ζ_err_max, find_ζ_aθ(0.0, ctrl)), length=24)
a_θ_vals = hcat([[a_θ(v_err, ζ_err, ctrl) for ζ_err in ζ_errs] for v_err in v_errs]...)

folder = joinpath(@__DIR__, "csv")
mkpath(folder)
df_heatmap = DataFrame(x = repeat(collect(v_errs), inner=length(ζ_errs)),
                        y = repeat(collect(ζ_errs), outer=length(v_errs)),
                        meta = vec(a_θ_vals))
CSV.write(joinpath(folder, "heatmap_aθ.csv"), df_heatmap)

fig_2d = heatmap(
    v_errs,
    ζ_errs,
    a_θ_vals,
    xlabel=L"$\|\tilde{v}_\varepsilon\|$",
    ylabel=L"$\|\tilde{\zeta}\|$",
    title=L"Heatmap of $a_θ$"
)

# Boundary of a_θ < 2c0/m0
v_errs = range(0.0, v_err_max, length=100)
ζ_err_aθ = [find_ζ_aθ(v_err, ctrl) for v_err in v_errs]
plot!(fig_2d, v_errs, ζ_err_aθ, lw=2, lc=:black, label=L"$a_θ = \frac{2 c_0}{m_0}$")

df_boundary_aθ = DataFrame(v_err = v_errs, ζ_err = ζ_err_aθ)
CSV.write(joinpath(folder, "boundary_aθ.csv"), df_boundary_aθ)

# Boundary of W1 < α_opt
ζ_err_W1 = [find_ζ_W1(0.0, v_err, α_opt, ctrl) for v_err in v_errs]
plot!(fig_2d, v_errs, ζ_err_W1, lw=2, lc=:green, label=L"$W_1 = \alpha$")

# Note that the boundary is an ellipse
ellfile = open(joinpath(@__DIR__, "csv", "ellipse.txt"), "w")
write(ellfile, "W1: v = $(v_err_max)*cos(t), ζ = $(ζ_err_max)*sin(t) \n")

# Boundary of W2 < α_opt
_, v_err_max, ζ_err_max = W2_limits(α_opt, ctrl)
write(ellfile, "W2: v = $(v_err_max)*cos(t), ζ = $(ζ_err_max)*sin(t) \n")
close(ellfile)
v_errs = range(0.0, v_err_max, length=100)
ζ_err_W2 = [find_ζ_W2(0.0, v_err, α_opt, ctrl) for v_err in v_errs]
plot!(fig_2d, v_errs, ζ_err_W2, lw=2, lc=:blue, label=L"$W_2 = \alpha$")

The region of convergence is somewhere within the set $\Omega_2$. Let's plot a 3D surface of the set:

In [ ]:
e_max, v_err_max, ζ_err_max = W2_limits(α_opt, ctrl)

v_grid = Float64[]
ζ_grid = Float64[]
e_grid = Float64[]

for θ in range(0.0, π/2, length=24)
    D = range(0.0, 1.0, length=24)
    v_err = D .* v_err_max * cos(θ)
    ζ_err = D .* ζ_err_max * sin(θ)
    e = [find_e_W2(v, ζ, α_opt, ctrl) for (v, ζ) in zip(v_err, ζ_err)]
    append!(v_grid, v_err)
    append!(ζ_grid, ζ_err)
    append!(e_grid, e)
end


e_grid[isnan.(e_grid)] .= 0.0
df_surface = DataFrame(x = v_grid,
                       y = ζ_grid,
                       z = e_grid)
CSV.write(joinpath(folder, "lyapunov_surface.csv"), df_surface)

fig_3d = scatter(
    v_grid,
    ζ_grid,
    e_grid,
    xlabel=L"$\|\tilde{v}_\varepsilon\|$",
    ylabel=L"$\|\tilde{\zeta}\|$",
    zlabel=L"$\|e\|$",
    title=L"Lyapunov function $W_2$ surface",
    camera=(70,50)
)